In [1]:
#| hide

%load_ext autoreload
%autoreload 2

# OpenAI Agent SDK -- No tools -- Parametric Knowledge

> Here, we ablate whether the best-performing model knows the dataset

- skip_showdoc: true
- skip_exec: true

In [2]:
#| default_exp experiments_no_tools

In [3]:
#| hide

import agents
from claimdb.configuration import *

## Prompts, I/O, Client

**OpenAI's agents handle structured outputs well with the Pydantic types**. They give the descriptions correctly. that is why in the prompts you will not see me explaining specific output types more. These comments are in the pydantic (`BaseModel` as `description`)

In [4]:
#| export
from pydantic import BaseModel, Field
from typing import Literal
from agents import Runner, Agent
import json

In [5]:
#| export
from claimdb.transformation import claim_collection_json_to_parsed

### I/O Schema

In [6]:
#| hide
#| export

class ClaimVerdict(BaseModel):
    verdict: Literal["ENTAILED", "CONTRADICTED", "NOT ENOUGH INFO"] = Field(
        ...,
        description="Whether the claim is supported, contradicted, or undecidable from the database."
    )
    justification: str = Field(
        ...,
        description="Brief justification (1-2 sentences) of the verdict."
    )

### Prompt

In [7]:
#| export

BASE_PROMPT_NO_TOOLS = f"""
You are a fact-checking assistant. You will be given a natural-language  claim and optional external information.

Your task is to determine whether the claim is "ENTAILED",  "CONTRADICTED", or "NOT ENOUGH INFO" based solely on what you know, without accessing any external tools or databases. The labels are defined as follows:

- ENTAILED: The claim is supported by your knowledge.
- CONTRADICTED: The claim is refuted by your knowledge.
- NOT ENOUGH INFO: You do not have sufficient knowledge to decide.

Do not ask the user for clarification or additional information.
"""

In [8]:
print(BASE_PROMPT_NO_TOOLS)


You are a fact-checking assistant. You will be given a natural-language  claim and optional external information.

Your task is to determine whether the claim is "ENTAILED",  "CONTRADICTED", or "NOT ENOUGH INFO" based solely on what you know, without accessing any external tools or databases. The labels are defined as follows:

- ENTAILED: The claim is supported by your knowledge.
- CONTRADICTED: The claim is refuted by your knowledge.
- NOT ENOUGH INFO: You do not have sufficient knowledge to decide.

Do not ask the user for clarification or additional information.



## OpenAI Agents

### Single Example Test

In [9]:
model = "gpt-5-nano"
model = "gpt-5-mini"
model_dir = config.experiments_dir_pub / 'no_tools' / model
model_dir.mkdir(parents=True, exist_ok=True)

In [10]:
with open(config.final_benchmark_dir / 'test-public.jsonl', "r") as f:
    all_claims = [json.loads(line) for line in f]

claim = all_claims[4]

In [11]:
claim

{'claim_id': 1334,
 'bird_id': 202,
 'db_name': 'toxicology',
 'label': 'CONTRADICTED',
 'claim': "There are exactly five triple type bonds (bond_type '#') in the dataset.",
 'extra_info': "triple type bonds refers to bond_type = '#'"}

In [12]:
fact_checker_agent = Agent(
    name="Fact-Checker",
    instructions=BASE_PROMPT_NO_TOOLS,
    model=model,
    tools=[],
    output_type=ClaimVerdict,
)
    

In [13]:
inp = f"Claim: {claim['claim']}\nExtra Information: {claim['extra_info']}"
#inp = f"Do you see what tools and metadata of tools you have?"

print(inp)

Claim: There are exactly five triple type bonds (bond_type '#') in the dataset.
Extra Information: triple type bonds refers to bond_type = '#'


In [14]:
result = await Runner.run(
    fact_checker_agent, 
    inp, 
    max_turns=1
)

In [15]:
result.final_output

ClaimVerdict(verdict='NOT ENOUGH INFO', justification="I have no access to the dataset referenced, so I cannot confirm the exact count of bonds with bond_type '#'. Additional data or counts from the dataset are needed to verify the claim.")

In [16]:
claim['label']

'CONTRADICTED'

In [17]:
result.to_input_list()

[{'content': "Claim: There are exactly five triple type bonds (bond_type '#') in the dataset.\nExtra Information: triple type bonds refers to bond_type = '#'",
  'role': 'user'},
 {'id': 'rs_08e1df88cc4301e700699376c56ba8819db8ddeac2b72c62dc',
  'summary': [],
  'type': 'reasoning'},
 {'id': 'msg_08e1df88cc4301e700699376c61360819da20a0f367598861f',
  'content': [{'annotations': [],
    'text': '{"verdict":"NOT ENOUGH INFO","justification":"I have no access to the dataset referenced, so I cannot confirm the exact count of bonds with bond_type \'#\'. Additional data or counts from the dataset are needed to verify the claim."}',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

### Agent's RunResult to dict

#### Example to understand RunResult

`to_input_list()` is the complete pipeline of all things that happened in json and text

In [18]:
lst = result.to_input_list()

In [19]:
result.to_input_list()[-1]

{'id': 'msg_08e1df88cc4301e700699376c61360819da20a0f367598861f',
 'content': [{'annotations': [],
   'text': '{"verdict":"NOT ENOUGH INFO","justification":"I have no access to the dataset referenced, so I cannot confirm the exact count of bonds with bond_type \'#\'. Additional data or counts from the dataset are needed to verify the claim."}',
   'type': 'output_text',
   'logprobs': []}],
 'role': 'assistant',
 'status': 'completed',
 'type': 'message'}

Usage (see [here](https://github.com/openai/openai-agents-python/blob/f903ad0ac44e1c5c959301bd3c8721fbd4cd4e5b/examples/basic/usage_tracking.py#L41))

#### Function Definition

In [20]:
#| export

def run_result_to_dict(result, ollama=False) -> dict:
    """Convert an Agent's RunResult to a dictionary."""
    info_dict = {}

    if isinstance(result, Exception):
        return {
            'verdict': "",
            'error': str(result),
            'justification': "",
            'model_name': "",
            'model_settings': "",
            'usage': [],
            'to_input_list': []
        }

    # 1. Final output
    info_dict['verdict'] = result.final_output.verdict
    info_dict['justification'] = result.final_output.justification
    info_dict['final_output'] = str(result.final_output)

    # 2. Model Settings
    info_dict['model_name'] = result._last_agent.model
    if ollama: info_dict['model_name'] = info_dict['model_name'].model
    info_dict['model_settings'] = result._last_agent.model_settings.to_json_dict()

    # 3. All Requests Costs (the total is the sum)
    usage = []
    for request_usage in result.context_wrapper.usage.request_usage_entries:
        cached_input_tokens = request_usage.input_tokens_details.cached_tokens
        regular_input_tokens = request_usage.input_tokens - cached_input_tokens
        output_tokens = request_usage.output_tokens

        usage.append(
            {
                "regular_input_tokens": regular_input_tokens,
                "cached_input_tokens": cached_input_tokens,
                "output_tokens": output_tokens,
            }
        )
    info_dict['usage'] = usage

    # 4. The complete Agentic Pipeline
    info_dict['to_input_list'] = result.to_input_list()

    return info_dict

### Run OpenAI models on All Claims

Here, simply change **model** name and run this subsection of the notebook again and again!

In [21]:
#| export
import asyncio
import random

In [22]:
#| export
def return_coroutines(test_claims, model):
    cors = []
    claim_ids = []

    for claim in test_claims:

        fact_checker_agent = Agent(
            name="Fact-Checker",
            instructions=BASE_PROMPT_NO_TOOLS,
            model=model,
            tools=[],
            output_type=ClaimVerdict,
        )

        inp = f"Claim: {claim['claim']}\nExtra Information: {claim['extra_info']}"

        cors.append(Runner.run(fact_checker_agent, inp, max_turns=20))
        
        claim_ids.append(claim['claim_id'])
    
    return cors, claim_ids

In [23]:
#| export

#TODO: tool returns exception without messing up the dependancies on 07b.

#model = "gpt-5-mini"
#model = "gpt-4.1-nano"
model = "gpt-5-mini"

batch_size = 100

results_path = config.experiments_dir_pub / 'no_tools' / f"{model}.jsonl"
results_path.touch()

In [24]:
#| export
import asyncio

In [25]:
#| export
with open(results_path, 'r') as f:
    already_tested = [json.loads(line)['claim_id'] for line in f]

benchmark = []
with open(config.final_benchmark_dir / 'test-public.jsonl') as f:
    for line in f: 
        parsed_claim = json.loads(line)
        if parsed_claim['claim_id'] in already_tested: continue
        benchmark.append(parsed_claim)

In [26]:
len(benchmark)

1000

In [27]:
#| export
async def run_tests():

    for i in range(0, len(benchmark), batch_size):
    #for i in range(0, 10, batch_size):
        test_claims = benchmark[i:i+batch_size]

        cors, claim_ids = return_coroutines(test_claims, model)

        results = await asyncio.gather(*cors, return_exceptions=True)

        for claim_id, res in zip(claim_ids, results):
            results_dict = {'claim_id': claim_id} | run_result_to_dict(res)
            results_path.open('a').write(json.dumps(results_dict) + '\n')

In [28]:
await run_tests()

In [29]:
#| export 
try: from nbdev.imports import IN_NOTEBOOK
except: IN_NOTEBOOK=False

In [30]:
#| export
if __name__ == "__main__" and not IN_NOTEBOOK:
    print(f"#Exps Left: {len(benchmark)}")
    print(model)
    asyncio.run(run_tests())

## End

In [31]:
#| hide
import nbdev; nbdev.nbdev_export()